# Land-atmosphere lag coupling in the hindcasts

At each land point, the correlation between the land-side variable on day *t* and the atmosphere-side variable on day *t + lag*, computed from **departures of each member from the ensemble mean of that day** and pooled over members and days. Removing the ensemble mean removes what all members share (the seasonal drift over the hindcast, the initial shock), so *r* measures how member-to-member land differences go with atmosphere differences. Positive lag: the land variable leads. An asymmetric curve shows which side drives the other: for soil moisture vs PRECT, lag > 0 is the soil-moisture feedback on rainfall and lag < 0 is rainfall wetting the soil.

Daily means come from the post-processed series via `util.model_daily`: atmosphere `ts/daily_mean` (00Z-24Z), land `ts/daily` (ELM). `SOILM_5CM` is the thickness-weighted mean of the soil layers in the top 5 cm. Land points: land fraction ≥ 0.5, north of 60°S.

Cases: **S0** hindcasts start from the Dec 2011 analyses and cover Jan-Feb 2012; **S1** hindcasts start from the May 2012 analyses and cover Jun-Jul 2012.

## 1. Settings

In [ ]:
from pathlib import Path

INPUT_DATA_ROOT = Path("/compyfs/zhan391/v3_dart_cda_scratch")
LAND_SUPPORT_DIR = Path("/compyfs/zhan391/acme_init/lnd_sea_mask")   # land mask, soil-layer thickness
WORK_DIR = Path("/compyfs/zhan391/work_package/e3sm_dart_analysis")
DIAGNOSTIC_OUTPUT_ROOT = Path("/compyfs/www/zhan391/e3sm_dart/diag_dart_2026")
workflow_name = "fcst_lac_feedback"

run_segment = "fc"
# season -> experiments (None: every hindcast of the season)
seasons = {"S0": None, "S1": None}
# (land-side variable, atmosphere-side variable); see util.lnd_atm_feedback.VARIABLES
pairs = (("SOILM_5CM", "LHFLX"), ("SOILM_5CM", "PRECT"), ("TS", "LHFLX"))
lags = tuple(range(-10, 11))  # days; the land variable leads for lag > 0
map_lags = (0, 1)
skip_days = 0  # leave out the first days of each hindcast (initial shock)
regions = ("land", "CONUS", "NHMidLat", "Tropics", "MAR_CONT")
alpha = 0.05
n_boot = 500
block_days = 5  # bootstrap block length (days) for the regional-curve intervals
max_members = None  # e.g. 2 for a quick test
force_compute = False
show = True
FIGURE_DPI = 200

## 2. Project setup

In [ ]:
import sys
import matplotlib.pyplot as plt

repo = WORK_DIR.resolve()
if str(repo / "diagnostic") not in sys.path:
    sys.path.insert(0, str(repo / "diagnostic"))
from configs.output_paths import workflow_output_dirs
from util.bias_metrics import experiment_registry
from util.lnd_atm_feedback import (
    REGIONS, VARIABLES, cache_feedback, feedback_cache_path, hindcast_days,
    plot_lag_curves, plot_lag_maps,
)

data_dir, figure_dir = workflow_output_dirs(workflow_name, output_root=DIAGNOSTIC_OUTPUT_ROOT)
unknown = set(regions) - set(REGIONS)
if unknown:
    raise ValueError(f"Unknown regions {sorted(unknown)}; choose from {list(REGIONS)}")
unknown = {v for pair in pairs for v in pair} - set(VARIABLES)
if unknown:
    raise ValueError(f"Unknown variables {sorted(unknown)}; choose from {list(VARIABLES)}")
if not set(map_lags) <= set(lags):
    raise ValueError("map_lags must be among lags")

## 3. Compute or reuse coupling caches

In [ ]:
processed = {}  # (season, x, y) -> {experiment: Dataset}
failures = []
for season, chosen in seasons.items():
    registry = experiment_registry(INPUT_DATA_ROOT, season, run_segment)
    experiments = list(registry) if chosen is None else list(chosen)
    for x_variable, y_variable in pairs:
        results = processed.setdefault((season, x_variable, y_variable), {})
        for experiment in experiments:
            config = registry[experiment]
            target = feedback_cache_path(data_dir, experiment, x_variable, y_variable, config["period"], skip_days)
            print(f"{season} {experiment}: {x_variable} vs {y_variable}")
            try:
                ds, written = cache_feedback(
                    target, force_compute=force_compute, lags=list(lags),
                    config=config, x_variable=x_variable, y_variable=y_variable,
                    days=hindcast_days(config["period"], skip_days),
                    landmask_file=LAND_SUPPORT_DIR / "landmask_1x1.nc", land_support_dir=LAND_SUPPORT_DIR,
                    regions=regions, alpha=alpha, n_boot=n_boot, block_days=block_days,
                    max_members=max_members,
                )
            except (FileNotFoundError, ValueError) as error:
                failures.append((season, experiment, x_variable, y_variable, str(error)))
                print("  [SKIP]", error)
                continue
            print("  [WRITE]" if written else "  [REUSE]", target)
            results[experiment] = ds
if failures:
    print(f"{len(failures)} request(s) failed; see `failures`")

## 4. Plot, save, and show

In [ ]:
for (season, x_variable, y_variable), results in processed.items():
    if not results:
        continue
    plotted = [r for r in regions if all(r in ds.region.values for ds in results.values())]
    stem = f"{season}_{x_variable}_vs_{y_variable}_skip{skip_days}d"
    fig = plot_lag_curves(results, plotted, title=f"{season}: {x_variable}(t) vs {y_variable}(t + lag), ensemble departures")
    output = figure_dir / f"{stem}_lag_curves.png"
    fig.savefig(output, dpi=FIGURE_DPI, bbox_inches="tight")
    print("[FIGURE]", output)
    if not show:
        plt.close(fig)
    for lag in map_lags:
        fig = plot_lag_maps(results, lag, alpha=alpha, title=f"{season}: {x_variable} vs {y_variable}")
        output = figure_dir / f"{stem}_lag{lag:+d}_maps.png"
        fig.savefig(output, dpi=FIGURE_DPI, bbox_inches="tight")
        print("[FIGURE]", output)
        if not show:
            plt.close(fig)

## Notes: method and cached fields

- **Departures**: member minus the ensemble mean of the same day, so experiments with 10 and 40 members measure the same quantity; at least 2 members are needed.
- **Significance (maps)**: two-sided t-test of the pooled *r* with the sample size reduced for the lag-1 autocorrelations *a* of the two departure series, `n_eff = n (1 - a_x a_y) / (1 + a_x a_y)`. Dots mark p < `alpha` (every other grid point). No correction for testing many grid points is applied.
- **Regional curves**: cos-lat weighted means of *r* over the region's land points; the shading is the `alpha` percentile interval of a circular block bootstrap over days (all members kept). The interval can be asymmetric when a few high-spread days carry much of the coupling.
- **Data**: daily means only; a variable available only as 00Z states is rejected. `skip_days` leaves out the first hindcast days.

Caches: `<experiment>_<x>_vs_<y>_<period>_skip<N>d_feedback.nc` with `r`, `p_value`, `n_eff` (lag, lat, lon) and `curve`, `curve_low`, `curve_high` (lag, region). A cache is rebuilt when its version or lags differ; set `force_compute = True` after the input files change.